# 3.3 Memory bounded heuristic search: IDA*, RBFS and SMA*: practice notebook

A* finds the cheapest path but keeps every node in memory. IDA*, RBFS and SMA* keep it optimal with linear or fixed memory, by repeating some work. Step through each one on the Romania map.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### IDA* on the Romania map

In [ ]:
# IDA* on the western part of the Romania map (enough for Arad to Bucharest).
roads = {
    "Arad": [("Zerind", 75), ("Sibiu", 140), ("Timisoara", 118)],
    "Zerind": [("Arad", 75), ("Oradea", 71)],
    "Oradea": [("Zerind", 71), ("Sibiu", 151)],
    "Timisoara": [("Arad", 118), ("Lugoj", 111)],
    "Lugoj": [("Timisoara", 111)],
    "Sibiu": [("Arad", 140), ("Oradea", 151), ("Fagaras", 99), ("Rimnicu Vilcea", 80)],
    "Fagaras": [("Sibiu", 99), ("Bucharest", 211)],
    "Rimnicu Vilcea": [("Sibiu", 80), ("Pitesti", 97), ("Craiova", 146)],
    "Craiova": [("Rimnicu Vilcea", 146), ("Pitesti", 138)],
    "Pitesti": [("Rimnicu Vilcea", 97), ("Craiova", 138), ("Bucharest", 101)],
    "Bucharest": [("Fagaras", 211), ("Pitesti", 101)],
}
h = {"Arad": 366, "Zerind": 374, "Oradea": 380, "Timisoara": 329, "Lugoj": 244, "Sibiu": 253, "Fagaras": 176,
     "Rimnicu Vilcea": 193, "Craiova": 160, "Pitesti": 100, "Bucharest": 0}
INF = 10 ** 9

def contour(city, g, limit, path):
    # depth first search that stops when f = g + h is above the limit.
    # It returns the path if it finds Bucharest, and otherwise the smallest f that was over the limit.
    f = g + h[city]
    if f > limit:
        return None, f
    if city == "Bucharest":
        return path, f
    smallest = INF
    for nxt, cost in roads[city]:
        if nxt not in path:
            found, over = contour(nxt, g + cost, limit, path + [nxt])
            if found is not None:
                return found, over
            smallest = min(smallest, over)
    return None, smallest

# We start from Sibiu to keep the run short. Try "Arad" too: the limits are then 366, 393, 413, 415, 417, 418.
limit = h["Sibiu"]
while True:
    print("iteration with f limit", limit)
    found, next_limit = contour("Sibiu", 0, limit, ["Sibiu"])
    if found is not None:
        print("found", found, "with cost", next_limit)
        break
    limit = next_limit

### RBFS on the Romania map

In [ ]:
# Recursive best first search (RBFS) on the same map.
roads = {
    "Arad": [("Zerind", 75), ("Sibiu", 140), ("Timisoara", 118)],
    "Zerind": [("Arad", 75), ("Oradea", 71)],
    "Oradea": [("Zerind", 71), ("Sibiu", 151)],
    "Timisoara": [("Arad", 118), ("Lugoj", 111)],
    "Lugoj": [("Timisoara", 111)],
    "Sibiu": [("Arad", 140), ("Oradea", 151), ("Fagaras", 99), ("Rimnicu Vilcea", 80)],
    "Fagaras": [("Sibiu", 99), ("Bucharest", 211)],
    "Rimnicu Vilcea": [("Sibiu", 80), ("Pitesti", 97), ("Craiova", 146)],
    "Craiova": [("Rimnicu Vilcea", 146), ("Pitesti", 138)],
    "Pitesti": [("Rimnicu Vilcea", 97), ("Craiova", 138), ("Bucharest", 101)],
    "Bucharest": [("Fagaras", 211), ("Pitesti", 101)],
}
h = {"Arad": 366, "Zerind": 374, "Oradea": 380, "Timisoara": 329, "Lugoj": 244, "Sibiu": 253, "Fagaras": 176,
     "Rimnicu Vilcea": 193, "Craiova": 160, "Pitesti": 100, "Bucharest": 0}
INF = 10 ** 9

def rbfs(city, g, f_city, f_limit, path):
    print("at", city, " f =", f_city, " limit =", f_limit)
    if city == "Bucharest":
        return path, f_city
    kids = []
    for nxt, cost in roads[city]:
        if nxt not in path:
            f_new = max(g + cost + h[nxt], f_city)     # a child is never better than its parent
            kids.append([f_new, nxt, g + cost])
    if len(kids) == 0:
        return None, INF
    while True:
        kids.sort()
        best = kids[0]
        if best[0] > f_limit:
            return None, best[0]                      # give up here and report the best f we saw
        alt = INF
        if len(kids) > 1:
            alt = kids[1][0]                          # f of the best alternative
        result, best[0] = rbfs(best[1], best[2], best[0], min(f_limit, alt), path + [best[1]])
        if result is not None:
            return result, best[0]

path, cost = rbfs("Arad", 0, h["Arad"], INF, ["Arad"])
print(path, cost)

## Your turn

### Task 1: Write IDA*

Write `ida_star(graph, h, start, goal)`. graph maps a node to a list of [child, cost]. Return [cost, limits], where limits is the list of f limits used, the last being the one where the goal is found. Return None if the goal cannot be reached.

Hint: Write a helper that returns (cost, f) when it finds the goal and (None, smallest f over the limit) otherwise. Loop until the limit becomes "infinity".

In [ ]:
def ida_star(graph, h, start, goal):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("ida_star({'S': [['A', 1], ['B', 4]], 'A': [['B', 2], ['C', 5]], 'B': [['C', 1]], 'C': [['G', 3]], 'G': []}, {'S': 6, 'A': 5, 'B': 3, 'C': 2, 'G': 0}, 'S', 'G')", [7, [6, 7]]), ("ida_star({'S': [['n1', 10], ['n3', 100]], 'n1': [['n2', 10]], 'n2': [['Goal', 10]], 'n3': [['Goal', 10]], 'Goal': []}, {'S': 30, 'n1': 20, 'n2': 10, 'n3': 10, 'Goal': 0}, 'S', 'Goal')", [30, [30]]), ("ida_star({'A': [['B', 1]], 'B': []}, {'A': 0, 'B': 0, 'C': 0}, 'A', 'C')", None)])

### Task 2: Write RBFS

Write `rbfs_cost(graph, h, start, goal)` that returns the cost of the path RBFS finds (or None). graph maps a node to a list of [child, cost]. Use f(child) = max(g + h(child), f(parent)), the f-limit min(limit, second best f), and back up the f of a child when it fails.

Hint: Keep the children as [f, name, g] lists so you can change f in place when a child backs up a value.

In [ ]:
def rbfs_cost(graph, h, start, goal):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("rbfs_cost({'S': [['A', 1], ['B', 4]], 'A': [['B', 2], ['C', 5]], 'B': [['C', 1]], 'C': [['G', 3]], 'G': []}, {'S': 6, 'A': 5, 'B': 3, 'C': 2, 'G': 0}, 'S', 'G')", 7), ("rbfs_cost({'S': [['n1', 10], ['n3', 100]], 'n1': [['n2', 10]], 'n2': [['Goal', 10]], 'n3': [['Goal', 10]], 'Goal': []}, {'S': 30, 'n1': 20, 'n2': 10, 'n3': 10, 'Goal': 0}, 'S', 'Goal')", 30), ("rbfs_cost({'A': [['B', 1]], 'B': []}, {'A': 0, 'B': 0, 'C': 0}, 'A', 'C')", None)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Write IDA*

On the small graph, the first limit is h(S) = 6. The smallest f over 6 is 7, and the goal is found at 7.

In [ ]:
def ida_star(graph, h, start, goal):
    big = 10 ** 9

    def search(node, g, limit, path):
        f = g + h[node]
        if f > limit:
            return None, f
        if node == goal:
            return g, f
        smallest = big
        for child, cost in graph[node]:
            if child not in path:
                found, over = search(child, g + cost, limit, path + [child])
                if found is not None:
                    return found, over
                smallest = min(smallest, over)
        return None, smallest

    limit = h[start]
    limits = []
    while limit < big:
        limits.append(limit)
        found, nxt = search(start, 0, limit, [start])
        if found is not None:
            return [found, limits]
        limit = nxt
    return None

In [ ]:
check([("ida_star({'S': [['A', 1], ['B', 4]], 'A': [['B', 2], ['C', 5]], 'B': [['C', 1]], 'C': [['G', 3]], 'G': []}, {'S': 6, 'A': 5, 'B': 3, 'C': 2, 'G': 0}, 'S', 'G')", [7, [6, 7]]), ("ida_star({'S': [['n1', 10], ['n3', 100]], 'n1': [['n2', 10]], 'n2': [['Goal', 10]], 'n3': [['Goal', 10]], 'Goal': []}, {'S': 30, 'n1': 20, 'n2': 10, 'n3': 10, 'Goal': 0}, 'S', 'Goal')", [30, [30]]), ("ida_star({'A': [['B', 1]], 'B': []}, {'A': 0, 'B': 0, 'C': 0}, 'A', 'C')", None)])

### Solution to task 2: Write RBFS

This is the same RBFS as the lesson, returning the cost instead of the path. When no path exists, the best f becomes "infinity" (big) and RBFS stops.

In [ ]:
def rbfs_cost(graph, h, start, goal):
    big = 10 ** 9

    def rbfs(node, g, f_node, limit, path):
        if node == goal:
            return g, f_node
        kids = []
        for child, cost in graph[node]:
            if child not in path:
                kids.append([max(g + cost + h[child], f_node), child, g + cost])
        if len(kids) == 0:
            return None, big
        while True:
            kids.sort()
            best = kids[0]
            if best[0] > limit or best[0] >= big:     # over the limit, or every child is a dead end
                return None, best[0]
            alt = big
            if len(kids) > 1:
                alt = kids[1][0]
            result, best[0] = rbfs(best[1], best[2], best[0], min(limit, alt), path + [best[1]])
            if result is not None:
                return result, best[0]

    cost, f = rbfs(start, 0, h[start], big, [start])
    return cost

In [ ]:
check([("rbfs_cost({'S': [['A', 1], ['B', 4]], 'A': [['B', 2], ['C', 5]], 'B': [['C', 1]], 'C': [['G', 3]], 'G': []}, {'S': 6, 'A': 5, 'B': 3, 'C': 2, 'G': 0}, 'S', 'G')", 7), ("rbfs_cost({'S': [['n1', 10], ['n3', 100]], 'n1': [['n2', 10]], 'n2': [['Goal', 10]], 'n3': [['Goal', 10]], 'Goal': []}, {'S': 30, 'n1': 20, 'n2': 10, 'n3': 10, 'Goal': 0}, 'S', 'Goal')", 30), ("rbfs_cost({'A': [['B', 1]], 'B': []}, {'A': 0, 'B': 0, 'C': 0}, 'A', 'C')", None)])